In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import json
import gseapy as gp

In [2]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

# Loading

In [3]:
def load(d):
    with open(f"../output/{d}/leiden_results/result_communities_selected.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"../output/{d}/leiden_results/result_communities_HGNC_selected.pkl", "rb") as f:
        communities_HGNC = pickle.load(f)
    # with open(f"../output/{d}/leiden_results/result_graph.pkl", "rb") as f:
    #     graph = pickle.load(f)    
    with open(f"../output/{d}/gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
        
    return communities,communities_HGNC,gene_to_index_distinct

In [4]:
communities_selected,communities_HGNC_selected,gene_to_index_distinct = load(DISEASE)

# index to HGNC

In [5]:
index_to_gene_distinct = {v: u for (u,v) in gene_to_index_distinct.items()}

In [6]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

# Community deepdive

In [7]:
comm_idx = 7

In [8]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [9]:
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1120,Ubiquitin Ligase-Substrate Adaptor Activity (G...,16/44,5.987088e-08,['molecular adaptor activity'],GO_Molecular_Function_2023,7.919429e-10,0.0,0.0,9.757764,204.488891,KLHDC2;KLHDC3;FEM1A;KLHL25;PEF1;FEM1B;PDCD6;FB...,0.363636
1,0,1120,Cysteine-Type Deubiquitinase Activity (GO:0004...,21/98,4.122296e-06,['catalytic activity'],GO_Molecular_Function_2023,8.724436e-08,0.0,0.0,4.666143,75.846065,OTUD4;OTUB2;USP36;USP47;USP25;USP48;USP6;USP49...,0.214286
2,1,1129,G Protein-Coupled Peptide Receptor Activity (G...,39/77,8.638396e-27,['molecular transducer activity'],GO_Molecular_Function_2023,1.872823e-28,0.0,0.0,17.732665,1132.140880,OPRD1;OXTR;VIPR1;VIPR2;NPR1;MLNR;OPRL1;FPR3;GP...,0.506494
3,1,1129,Anterior/Posterior Pattern Specification (GO:0...,28/59,3.548246e-17,['multicellular organismal process'],GO_Biological_Process_2023,8.213533e-20,0.0,0.0,15.455744,679.216871,GATA4;HHEX;SIX2;HOXA3;HOXC5;HOXC4;HES3;HOXC9;H...,0.474576
4,1,1129,Voltage-Gated Potassium Channel Complex (GO:00...,30/73,1.127232e-16,['protein-containing complex'],GO_Cellular_Component_2023,6.515793e-19,0.0,0.0,11.952515,500.510224,DPP10;KCNG1;HCN4;KCNC1;LRRC38;KCNA1;KCNC4;KCNA...,0.410959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
672,10,238,Exocytosis (GO:0006887),10/86,1.063367e-06,"['localization', 'cellular process']",GO_Biological_Process_2023,7.565454e-08,0.0,0.0,11.360803,186.284096,NSF;RAB10;SNAP47;VAMP7;TMED10;STX19;MIA3;SNAP2...,0.116279
673,10,238,Golgi Cisternae Pericentriolar Stack Reorganiz...,5/14,2.396149e-06,['Cell Cycle'],Reactome_2022,4.196644e-07,0.0,0.0,47.098236,691.581561,GOLGA2;RAB1A;RAB1B;GORASP1;USO1,0.357143
674,10,238,Protein Localization To Endoplasmic Reticulum ...,4/6,3.833169e-06,['localization'],GO_Biological_Process_2023,2.878665e-07,0.0,0.0,168.888889,2543.596563,SEC16B;SEC16A;GBF1;MIA3,0.666667
675,10,238,"Antigen Presentation: Folding, Assembly, Pepti...",6/28,4.461827e-06,['Immune System'],Reactome_2022,8.066580e-07,0.0,0.0,23.205329,325.579263,SEC23A;SEC24B;SEC24A;SEC24D;SEC24C;SEC31A,0.214286


In [10]:
go_df_filtered = important_terms[important_terms["Community Index"] == comm_idx]

In [11]:
go_df_filtered = go_df_filtered.sort_values(by = ["Adjusted P-value"], ascending = [True])

In [12]:
go_df_filtered

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
306,7,754,VEGFA-VEGFR2 Pathway R-HSA-4420097,62/93,3.506578e-64,['Signal Transduction'],Reactome_2022,6.786926e-66,0.0,0.0,55.534682,8333.291043,CYFIP2;PRR5;CYFIP1;NCKAP1;AHCYL1;NCF1;ITGB3;BR...,0.666667
307,7,754,EPH-Ephrin Signaling R-HSA-2682334,60/91,1.211928e-61,['Developmental Biology'],Reactome_2022,2.475982e-63,0.0,0.0,53.588361,7725.095772,EPHB6;ARPC1B;ARPC1A;ITSN1;CLTB;WASL;ACTG1;SYNG...,0.659341
308,7,754,Fcgamma Receptor (FCGR) Dependent Phagocytosis...,58/87,5.199764e-60,['Immune System'],Reactome_2022,1.118229e-61,0.0,0.0,55.221264,7750.080509,CYFIP2;CYFIP1;NCKAP1;AHCYL1;WIPF1;ARPC1B;WIPF2...,0.666667
309,7,754,Bacterial invasion of epithelial cells,54/77,5.189699e-58,['Infectious disease: bacterial'],KEGG_2021_Human,2.192831e-59,0.0,0.0,64.474658,8708.419783,ITGB1;ARPC1B;ARPC1A;ARPC5L;CLTB;ILK;PIK3CD;WAS...,0.701299
310,7,754,ErbB signaling pathway,53/85,8.316038e-53,['Signal transduction'],KEGG_2021_Human,4.685092e-54,0.0,0.0,45.396844,5574.514944,GSK3B;CDKN1B;PIK3CD;CBLB;AREG;ELK1;CRKL;NCK2;N...,0.623529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,7,754,Negative Regulation Of Signaling (GO:0023057),9/28,5.939201e-06,['biological regulation'],GO_Biological_Process_2023,5.316013e-07,0.0,0.0,12.224868,176.617211,BRAP;PPP3CA;PTPN1;PPP3CB;MPIG6B;SRMS;ERBB3;INP...,0.321429
549,7,754,CD28 Dependent Vav1 Pathway R-HSA-389359,6/12,7.373627e-06,['Immune System'],Reactome_2022,2.140730e-06,0.0,0.0,25.721925,335.783360,CDC42;LCK;FYN;PKN1;ARHGEF7;VAV1,0.500000
550,7,754,Positive Regulation Of Microtubule Polymerizat...,9/29,8.201933e-06,['biological regulation'],GO_Biological_Process_2023,7.451303e-07,0.0,0.0,11.613020,163.856308,PAK1;OCLN;CAV3;FES;RAC1;MAPT;MET;GIT1;CDK5R1,0.310345
551,7,754,Synthesis Of PIPs At Plasma Membrane R-HSA-166...,11/51,8.340729e-06,['Metabolism'],Reactome_2022,2.439439e-06,0.0,0.0,7.108546,91.869023,INPP5D;INPPL1;PTEN;PIK3R3;PIK3R2;PIP5K1B;PIK3R...,0.215686


In [13]:
go_df_filtered[go_df_filtered["Gene_set"] == "Reactome_2022"]

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
306,7,754,VEGFA-VEGFR2 Pathway R-HSA-4420097,62/93,3.506578e-64,['Signal Transduction'],Reactome_2022,6.786926e-66,0.0,0.0,55.534682,8333.291043,CYFIP2;PRR5;CYFIP1;NCKAP1;AHCYL1;NCF1;ITGB3;BR...,0.666667
307,7,754,EPH-Ephrin Signaling R-HSA-2682334,60/91,1.211928e-61,['Developmental Biology'],Reactome_2022,2.475982e-63,0.0,0.0,53.588361,7725.095772,EPHB6;ARPC1B;ARPC1A;ITSN1;CLTB;WASL;ACTG1;SYNG...,0.659341
308,7,754,Fcgamma Receptor (FCGR) Dependent Phagocytosis...,58/87,5.199764e-60,['Immune System'],Reactome_2022,1.118229e-61,0.0,0.0,55.221264,7750.080509,CYFIP2;CYFIP1;NCKAP1;AHCYL1;WIPF1;ARPC1B;WIPF2...,0.666667
311,7,754,RAC2 GTPase Cycle R-HSA-9013404,53/87,1.128707e-51,['Signal Transduction'],Reactome_2022,2.912792e-53,0.0,0.0,42.721994,5167.989784,ITGB1;CYFIP1;NCKAP1;DOCK3;TRIO;NCF1;ARHGAP1;BR...,0.609195
313,7,754,NRAGE Signals Death Thru JNK R-HSA-193648,43/57,2.700996e-48,['Signal Transduction'],Reactome_2022,8.712890e-50,0.0,0.0,83.079767,9385.060271,TRIO;ARHGEF26;MAGED1;RASGRF2;ITSN1;ARHGEF10L;K...,0.754386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,7,754,Cell Junction Organization R-HSA-446728,15/86,2.479720e-06,['Cell-Cell communication'],Reactome_2022,6.932552e-07,0.0,0.0,5.481808,77.742280,ITGB1;VASP;PRKCI;JUP;ACTN1;ILK;CDH5;AFDN;PARD6...,0.174419
537,7,754,MET Activates RAP1 And RAC1 R-HSA-8875555,6/11,3.895066e-06,['Signal Transduction'],Reactome_2022,1.105696e-06,0.0,0.0,30.867914,423.354536,RAP1B;RAP1A;DOCK7;RAPGEF1;CRK;CRKL,0.545455
546,7,754,CREB Phosphorylation R-HSA-199920,5/7,5.161452e-06,"['Immune System', 'Signal Transduction']",Reactome_2022,1.481836e-06,0.0,0.0,64.232310,862.140746,RPS6KA3;RPS6KA5;RPS6KA2;MAPKAPK2;RPS6KA1,0.714286
549,7,754,CD28 Dependent Vav1 Pathway R-HSA-389359,6/12,7.373627e-06,['Immune System'],Reactome_2022,2.140730e-06,0.0,0.0,25.721925,335.783360,CDC42;LCK;FYN;PKN1;ARHGEF7;VAV1,0.500000


In [14]:
print(go_df_filtered[["Term","Adjusted P-value","Overlap"]].to_latex(longtable=True,index=False,caption=f"Enriched terms for {DISEASE} Community {comm_idx}",label=f"appendix:{DISEASE}_{comm_idx}_term", float_format="%.2e",column_format="p{8cm}cc",))

\begin{longtable}{p{8cm}cc}
\caption{Enriched terms for BIPOLAR Community 7} \label{appendix:BIPOLAR_7_term} \\
\toprule
Term & Adjusted P-value & Overlap \\
\midrule
\endfirsthead
\caption[]{Enriched terms for BIPOLAR Community 7} \\
\toprule
Term & Adjusted P-value & Overlap \\
\midrule
\endhead
\midrule
\multicolumn{3}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
VEGFA-VEGFR2 Pathway R-HSA-4420097 & 3.51e-64 & 62/93 \\
EPH-Ephrin Signaling R-HSA-2682334 & 1.21e-61 & 60/91 \\
Fcgamma Receptor (FCGR) Dependent Phagocytosis R-HSA-2029480 & 5.20e-60 & 58/87 \\
Bacterial invasion of epithelial cells & 5.19e-58 & 54/77 \\
ErbB signaling pathway & 8.32e-53 & 53/85 \\
RAC2 GTPase Cycle R-HSA-9013404 & 1.13e-51 & 53/87 \\
Fc gamma R-mediated phagocytosis & 1.48e-48 & 53/97 \\
NRAGE Signals Death Thru JNK R-HSA-193648 & 2.70e-48 & 43/57 \\
RHOB GTPase Cycle R-HSA-9013026 & 6.98e-46 & 45/69 \\
Semaphorin Interactions R-HSA-373755 & 2.46e-41 & 41/64 \\
Protein Tyrosi

In [15]:
list(go_df_filtered["Term"])[:30]

['VEGFA-VEGFR2 Pathway R-HSA-4420097',
 'EPH-Ephrin Signaling R-HSA-2682334',
 'Fcgamma Receptor (FCGR) Dependent Phagocytosis R-HSA-2029480',
 'Bacterial invasion of epithelial cells',
 'ErbB signaling pathway',
 'RAC2 GTPase Cycle R-HSA-9013404',
 'Fc gamma R-mediated phagocytosis',
 'NRAGE Signals Death Thru JNK R-HSA-193648',
 'RHOB GTPase Cycle R-HSA-9013026',
 'Semaphorin Interactions R-HSA-373755',
 'Protein Tyrosine Kinase Activity (GO:0004713)',
 'Adherens junction',
 'Constitutive Signaling By Aberrant PI3K In Cancer R-HSA-2219530',
 'RHOQ GTPase Cycle R-HSA-9013406',
 'Oncogenic MAPK Signaling R-HSA-6802957',
 'Prolactin signaling pathway',
 'Signaling By SCF-KIT R-HSA-1433557',
 'Protein Phosphorylated Amino Acid Binding (GO:0045309)',
 'Choline metabolism in cancer',
 'VEGF signaling pathway',
 'Signaling By EGFR R-HSA-177929',
 'B cell receptor signaling pathway',
 'PD-L1 expression and PD-1 checkpoint pathway in cancer',
 'Regulation Of Phosphatidylinositol 3-Kinase Sign

In [16]:
go_df_filtered

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
306,7,754,VEGFA-VEGFR2 Pathway R-HSA-4420097,62/93,3.506578e-64,['Signal Transduction'],Reactome_2022,6.786926e-66,0.0,0.0,55.534682,8333.291043,CYFIP2;PRR5;CYFIP1;NCKAP1;AHCYL1;NCF1;ITGB3;BR...,0.666667
307,7,754,EPH-Ephrin Signaling R-HSA-2682334,60/91,1.211928e-61,['Developmental Biology'],Reactome_2022,2.475982e-63,0.0,0.0,53.588361,7725.095772,EPHB6;ARPC1B;ARPC1A;ITSN1;CLTB;WASL;ACTG1;SYNG...,0.659341
308,7,754,Fcgamma Receptor (FCGR) Dependent Phagocytosis...,58/87,5.199764e-60,['Immune System'],Reactome_2022,1.118229e-61,0.0,0.0,55.221264,7750.080509,CYFIP2;CYFIP1;NCKAP1;AHCYL1;WIPF1;ARPC1B;WIPF2...,0.666667
309,7,754,Bacterial invasion of epithelial cells,54/77,5.189699e-58,['Infectious disease: bacterial'],KEGG_2021_Human,2.192831e-59,0.0,0.0,64.474658,8708.419783,ITGB1;ARPC1B;ARPC1A;ARPC5L;CLTB;ILK;PIK3CD;WAS...,0.701299
310,7,754,ErbB signaling pathway,53/85,8.316038e-53,['Signal transduction'],KEGG_2021_Human,4.685092e-54,0.0,0.0,45.396844,5574.514944,GSK3B;CDKN1B;PIK3CD;CBLB;AREG;ELK1;CRKL;NCK2;N...,0.623529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548,7,754,Negative Regulation Of Signaling (GO:0023057),9/28,5.939201e-06,['biological regulation'],GO_Biological_Process_2023,5.316013e-07,0.0,0.0,12.224868,176.617211,BRAP;PPP3CA;PTPN1;PPP3CB;MPIG6B;SRMS;ERBB3;INP...,0.321429
549,7,754,CD28 Dependent Vav1 Pathway R-HSA-389359,6/12,7.373627e-06,['Immune System'],Reactome_2022,2.140730e-06,0.0,0.0,25.721925,335.783360,CDC42;LCK;FYN;PKN1;ARHGEF7;VAV1,0.500000
550,7,754,Positive Regulation Of Microtubule Polymerizat...,9/29,8.201933e-06,['biological regulation'],GO_Biological_Process_2023,7.451303e-07,0.0,0.0,11.613020,163.856308,PAK1;OCLN;CAV3;FES;RAC1;MAPT;MET;GIT1;CDK5R1,0.310345
551,7,754,Synthesis Of PIPs At Plasma Membrane R-HSA-166...,11/51,8.340729e-06,['Metabolism'],Reactome_2022,2.439439e-06,0.0,0.0,7.108546,91.869023,INPP5D;INPPL1;PTEN;PIK3R3;PIK3R2;PIP5K1B;PIK3R...,0.215686


In [17]:
print(list(go_df_filtered[go_df_filtered["Category"] == "['Signal Transduction']"]["Term"]))

['VEGFA-VEGFR2 Pathway R-HSA-4420097', 'RAC2 GTPase Cycle R-HSA-9013404', 'NRAGE Signals Death Thru JNK R-HSA-193648', 'RHOB GTPase Cycle R-HSA-9013026', 'RHOQ GTPase Cycle R-HSA-9013406', 'Signaling By SCF-KIT R-HSA-1433557', 'Signaling By EGFR R-HSA-177929', 'Extra-nuclear Estrogen Signaling R-HSA-9009391', 'Signaling By ERBB2 R-HSA-1227986', 'Signaling By Non-Receptor Tyrosine Kinases R-HSA-9006927', 'RHOU GTPase Cycle R-HSA-9013420', 'Nuclear Events (Kinase And Transcription Factor Activation) R-HSA-198725', 'RHO GTPases Activate ROCKs R-HSA-5627117', 'Signaling To ERKs R-HSA-187687', 'Signaling By MET R-HSA-6806834', 'Downstream Signal Transduction R-HSA-186763', 'RHOF GTPase Cycle R-HSA-9035034', 'VEGFR2 Mediated Vascular Permeability R-HSA-5218920', 'Insulin Receptor Signaling Cascade R-HSA-74751', 'Negative Regulation Of MAPK Pathway R-HSA-5675221', 'RND3 GTPase Cycle R-HSA-9696264', 'RHO GTPases Activate NADPH Oxidases R-HSA-5668599', 'Signaling By Erythropoietin R-HSA-9006335

In [18]:
A = ['Class A/1 (Rhodopsin-like Receptors) R-HSA-373076', 'GPCR Ligand Binding R-HSA-500792', 'Peptide Ligand-Binding Receptors R-HSA-375276', 'Signaling By GPCR R-HSA-372790', 'GPCR Downstream Signaling R-HSA-388396', 'G Alpha (I) Signaling Events R-HSA-418594', 'G Alpha (Q) Signaling Events R-HSA-416476', 'Amine Ligand-Binding Receptors R-HSA-375280', 'G Alpha (S) Signaling Events R-HSA-418555', 'Chemokine Receptors Bind Chemokines R-HSA-380108', 'Serotonin Receptors R-HSA-390666']

B = ['GPCR Ligand Binding R-HSA-500792', 'Signaling By GPCR R-HSA-372790', 'GPCR Downstream Signaling R-HSA-388396', 'Class A/1 (Rhodopsin-like Receptors) R-HSA-373076', 'Peptide Ligand-Binding Receptors R-HSA-375276', 'G Alpha (I) Signaling Events R-HSA-418594', 'G Alpha (S) Signaling Events R-HSA-418555', 'G Alpha (Q) Signaling Events R-HSA-416476', 'Class B/2 (Secretin Family Receptors) R-HSA-373080', 'Chemokine Receptors Bind Chemokines R-HSA-380108', 'Glucagon-type Ligand Receptors R-HSA-420092', 'Calcitonin-like Ligand Receptors R-HSA-419812', 'Lysosphingolipid And LPA Receptors R-HSA-419408', 'G Alpha (Z) Signaling Events R-HSA-418597', 'Formyl Peptide Receptors Bind Formyl Peptides And Many Other Ligands R-HSA-444473']

In [19]:
list(go_df_filtered["Category"].unique())

["['Signal Transduction']",
 "['Developmental Biology']",
 "['Immune System']",
 "['Infectious disease: bacterial']",
 "['Signal transduction']",
 "['Immune system']",
 "['catalytic activity']",
 "['Cellular community - eukaryotes']",
 "['Disease']",
 "['Endocrine system']",
 "['binding']",
 "['Cancer: overview']",
 '[]',
 "['cellular process', 'biological regulation']",
 "['Cancer: specific types']",
 "['biological regulation']",
 "['Hemostasis']",
 "['cellular process', 'response to stimulus', 'biological regulation']",
 "['Hemostasis', 'Signal Transduction']",
 "['Neuronal System']",
 "['Sensory system']",
 "['immune system process', 'cellular process', 'biological regulation']",
 "['Endocrine and metabolic disease']",
 "['cellular anatomical structure']",
 "['molecular transducer activity', 'catalytic activity']",
 "['Nervous system']",
 "['cellular process']",
 "['molecular function regulator activity']",
 "['Vesicle-mediated transport']",
 "['Immune System', 'Signal Transduction'